In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet("../data/processed/hdfs_sequences.parquet")

with open("../data/processed/template_map.json", 'r') as f:
    template_map = json.load(f)

print(f"Loaded {len(df):,} blocks")
print(f"Templates: {len(template_map)}")
print(f"Labels: {df['label'].value_counts().to_dict()}")

Loaded 575,061 blocks
Templates: 45
Labels: {'Normal': 558223, 'Anomaly': 16838}


In [2]:
all_event_ids = sorted(set(e for seq in df['sequence'] for e in seq))
print(f"Unique event IDs in data: {len(all_event_ids)}")

def sequence_to_counts(seq, event_ids):
    """Convert a sequence to a count vector."""
    counts = {eid: 0 for eid in event_ids}
    for e in seq:
        if e in counts:
            counts[e] += 1
    return [counts[eid] for eid in event_ids]

print("Building feature matrix...")
X = np.array([sequence_to_counts(seq, all_event_ids) for seq in df['sequence']])
y = (df['label'] == 'Anomaly').astype(int).values

print(f"Feature matrix shape: {X.shape}")
print(f"  Rows (blocks): {X.shape[0]:,}")
print(f"  Columns (event types): {X.shape[1]}")
print(f"Anomaly labels: {y.sum():,} anomalies, {(y==0).sum():,} normal")


Unique event IDs in data: 45
Building feature matrix...
Feature matrix shape: (575061, 45)
  Rows (blocks): 575,061
  Columns (event types): 45
Anomaly labels: 16,838 anomalies, 558,223 normal


In [3]:
from sklearn.model_selection import train_test_split

normal_mask = y == 0
anomaly_mask = y == 1

X_normal = X[normal_mask]
X_anomaly = X[anomaly_mask]

X_train, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)

X_test = np.vstack([X_test_normal, X_anomaly])
y_test = np.array([0] * len(X_test_normal) + [1] * len(X_anomaly))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set (normal only): {X_train.shape[0]:,}")
print(f"Test set: {X_test.shape[0]:,}")
print(f"  Normal in test: {(y_test==0).sum():,}")
print(f"  Anomaly in test: {(y_test==1).sum():,}")

Training set (normal only): 446,578
Test set: 128,483
  Normal in test: 111,645
  Anomaly in test: 16,838


In [4]:
print("Training Isolation Forest...")
iso_forest = IsolationForest(
    contamination=0.03,
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X_train_scaled)

y_pred_iso = iso_forest.predict(X_test_scaled)
y_pred_iso = (y_pred_iso == -1).astype(int)

print(f"\n--- Isolation Forest Results ---")
print(classification_report(y_test, y_pred_iso, target_names=['Normal', 'Anomaly']))

Training Isolation Forest...

--- Isolation Forest Results ---
              precision    recall  f1-score   support

      Normal       0.88      0.97      0.92    111645
     Anomaly       0.36      0.11      0.17     16838

    accuracy                           0.86    128483
   macro avg       0.62      0.54      0.55    128483
weighted avg       0.81      0.86      0.82    128483



In [5]:
print("Training One-Class SVM on sample (faster)...")

sample_size = 50_000
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_train_sample = X_train_scaled[sample_idx]

oc_svm = OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=0.03,
)
oc_svm.fit(X_train_sample)

y_pred_svm = oc_svm.predict(X_test_scaled)
y_pred_svm = (y_pred_svm == -1).astype(int)

print(f"\n--- One-Class SVM Results ---")
print(classification_report(y_test, y_pred_svm, target_names=['Normal', 'Anomaly']))

Training One-Class SVM on sample (faster)...

--- One-Class SVM Results ---
              precision    recall  f1-score   support

      Normal       1.00      0.24      0.39    111645
     Anomaly       0.17      1.00      0.29     16838

    accuracy                           0.34    128483
   macro avg       0.58      0.62      0.34    128483
weighted avg       0.89      0.34      0.38    128483



In [6]:
print("Training PCA model...")
pca = PCA(n_components=10, random_state=42)
pca.fit(X_train_scaled)

X_test_pca = pca.transform(X_test_scaled)
X_test_reconstructed = pca.inverse_transform(X_test_pca)

reconstruction_error = np.mean((X_test_scaled - X_test_reconstructed) ** 2, axis=1)

threshold = np.percentile(reconstruction_error[y_test == 0], 97)
y_pred_pca = (reconstruction_error > threshold).astype(int)

print(f"Threshold (97th percentile of normal): {threshold:.4f}")
print(f"\n--- PCA Reconstruction Results ---")
print(classification_report(y_test, y_pred_pca, target_names=['Normal', 'Anomaly']))

Training PCA model...
Threshold (97th percentile of normal): 0.0000

--- PCA Reconstruction Results ---
              precision    recall  f1-score   support

      Normal       1.00      0.98      0.99    111645
     Anomaly       0.90      1.00      0.94     16838

    accuracy                           0.98    128483
   macro avg       0.95      0.99      0.97    128483
weighted avg       0.99      0.98      0.98    128483

